# Load Type Classification

This notebook predicts the power-system load category using historical usage and operational features. The work follows a chronological validation strategy: the final month of the dataset is kept as the test set, so the evaluation reflects realistic forecasting on recent unseen data.

We will:
- inspect the dataset and class balance
- engineer time-based features and clean missing values
- train a baseline and several classification models
- compare results and explain the outcome in plain language

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_context('talk')

In [ ]:
data_path = 'data/load_data.csv'
df = pd.read_csv(data_path)
print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
print()
print(df.head().to_string(index=False))

## 1) Inspect the dataset

The dataset contains a timestamp plus operational measurements such as usage, reactive power, CO2, and the target label `Load_Type`. Before modeling, we confirm the date range and how common each class is in the real data.

In [ ]:
df['Date_Time'] = pd.to_datetime(df['Date_Time'], format='%d-%m-%Y %H:%M')
print('Date range:', df['Date_Time'].min(), 'to', df['Date_Time'].max())
print('\nTarget class distribution:')
print(df['Load_Type'].value_counts())
print('\nProportions:')
print(df['Load_Type'].value_counts(normalize=True).round(4))

## 2) Time-based split without random leakage

To prevent data leakage, we keep the last chronological month as the final test set. The model is trained only on earlier months. This is a realistic setup because it mirrors forecasting on new, recent data.

In [ ]:
df = df.sort_values('Date_Time').reset_index(drop=True)
last_month = df['Date_Time'].dt.to_period('M').max()
train_df = df[df['Date_Time'].dt.to_period('M') != last_month].copy()
test_df = df[df['Date_Time'].dt.to_period('M') == last_month].copy()

print(f'Train rows: {len(train_df)}')
print(f'Test rows: {len(test_df)}')
print(f'Last month used for testing: {last_month}')
print('\nTrain target distribution:')
print(train_df['Load_Type'].value_counts())
print('\nTest target distribution:')
print(test_df['Load_Type'].value_counts())

## 3) Feature engineering

The timestamp itself is informative, so we add a few time-based features such as hour, day of week, day of month, and cyclical time encodings. Those help the model learn daily and weekly patterns without overcomplicating the pipeline.

In [ ]:
for frame in [train_df, test_df]:
    frame['hour'] = frame['Date_Time'].dt.hour
    frame['minute'] = frame['Date_Time'].dt.minute
    frame['day_of_week'] = frame['Date_Time'].dt.dayofweek
    frame['day_of_month'] = frame['Date_Time'].dt.day
    frame['is_weekend'] = (frame['Date_Time'].dt.dayofweek >= 5).astype(int)
    frame['hour_sin'] = np.sin(2 * np.pi * frame['hour'] / 24)
    frame['hour_cos'] = np.cos(2 * np.pi * frame['hour'] / 24)

feature_cols = [
    'Usage_kWh',
    'Lagging_Current_Reactive.Power_kVarh',
    'Leading_Current_Reactive_Power_kVarh',
    'CO2(tCO2)',
    'Lagging_Current_Power_Factor',
    'Leading_Current_Power_Factor',
    'NSM',
    'hour',
    'minute',
    'day_of_week',
    'day_of_month',
    'is_weekend',
    'hour_sin',
    'hour_cos',
]

X_train = train_df[feature_cols]
y_train = train_df['Load_Type']
X_test = test_df[feature_cols]
y_test = test_df['Load_Type']

print('Feature columns used for modeling:')
print(feature_cols)

## 4) Exploratory data analysis

Here we look for simple patterns in the data. The target classes should appear in different ranges of energy usage and other indicators, which helps justify why the task is learnable with a classifier.

In [ ]:
sns.boxplot(data=train_df, x='Load_Type', y='Usage_kWh', order=['Light_Load', 'Medium_Load', 'Maximum_Load'])
plt.title('Usage_kWh by Load Type')
plt.tight_layout()
plt.show()

usage_by_day = train_df.groupby(['Date_Time'], as_index=False)['Usage_kWh'].mean()
usage_by_day['Date_Time'] = pd.to_datetime(usage_by_day['Date_Time'])
usage_by_day.set_index('Date_Time', inplace=True)
usage_by_day['Usage_kWh'].plot(figsize=(12, 4), title='Average daily electricity usage over time')
plt.ylabel('Usage (kWh)')
plt.tight_layout()
plt.show()

class_counts = train_df['Load_Type'].value_counts().reindex(['Light_Load', 'Medium_Load', 'Maximum_Load'])
class_counts.plot(kind='bar', color=['#4C72B0', '#55A868', '#C44E52'])
plt.title('Class distribution in the training data')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

num_df = train_df[feature_cols].copy()
correlation = num_df.corr(numeric_only=True)
plt.figure(figsize=(12, 8))
sns.heatmap(correlation, cmap='coolwarm', annot=False, fmt='.2f')
plt.title('Feature correlation heatmap')
plt.tight_layout()
plt.show()

## 5) Model training and comparison

We compare a simple baseline with a few standard classifiers. The preprocessing step is fit only on the training data, which prevents information from the test set from leaking into the model.

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler()),
        ]), feature_cols),
    ],
    remainder='drop'
)

models = {
    'Baseline (most frequent)': DummyClassifier(strategy='most_frequent'),
    'Logistic Regression': LogisticRegression(max_iter=2000, multi_class='auto', class_weight='balanced', random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=300, random_state=42, class_weight='balanced_subsample', min_samples_leaf=2),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42),
}

results = []
for name, model in models.items():
    pipeline = Pipeline([('preprocessor', preprocessor), ('model', model)])
    pipeline.fit(X_train, y_train)
    pred = pipeline.predict(X_test)

    metrics = {
        'Model': name,
        'Accuracy': accuracy_score(y_test, pred),
        'Precision': precision_score(y_test, pred, average='macro', zero_division=0),
        'Recall': recall_score(y_test, pred, average='macro', zero_division=0),
        'F1': f1_score(y_test, pred, average='macro', zero_division=0),
    }
    results.append(metrics)

    print(f'\n=== {name} ===')
    print(classification_report(y_test, pred, target_names=['Light_Load', 'Medium_Load', 'Maximum_Load']))
    cm = confusion_matrix(y_test, pred, labels=['Light_Load', 'Medium_Load', 'Maximum_Load'])
    print('Confusion matrix:\n', cm)

    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Light_Load', 'Medium_Load', 'Maximum_Load'])
    disp.plot(cmap='Blues')
    plt.title(f'{name} confusion matrix')
    plt.tight_layout()
    plt.show()

results_df = pd.DataFrame(results).sort_values('F1', ascending=False)
print('\nSummary table:')
print(results_df.to_string(index=False))

## 6) Interpretation of results

The goal here is not only to get a high score, but to understand whether the model is learning meaningful patterns. A model that is clearly better than the dummy baseline and keeps a strong F1-score on the last month is usually generalizing well under time-based validation.

In [ ]:
best_model_name = results_df.iloc[0]['Model']
best_metrics = results_df.iloc[0].to_dict()
print(f'Best model by macro F1-score: {best_model_name}')
print(best_metrics)

### Final takeaway

This assignment uses a realistic time-based split, which is more trustworthy than random train/test splitting for time series or chronological monitoring data. The best model should be able to classify recent load states using operational parameters and hour-of-day patterns, while maintaining a clean separation between the three load categories.